In [ ]:
library(dplyr)
library(ggplot2)
library(reshape2)

panel <- read.csv("../data/monthly_panel.csv")

print(colnames(panel))

In [ ]:
check_cols <- c("Binary_Artifacts_score", "Code_Review_score", "Contrib_score", 
                 "Dangerous_Workflow_score", "DependUpTool_score", "Maintained_score",
                 "Pinned_Dependencies_score", "Security_Policy_score", "Token_Permissions_score")


df_demeaned <- panel %>%
  group_by(package_name) %>%
  mutate(across(all_of(check_cols), 
                ~ .x - mean(.x, na.rm = TRUE), 
                .names = "demean_{.col}")) %>%
  ungroup()

demean_cols <- paste0("demean_", check_cols)

co_implementation_matrix <- cor(df_demeaned[, demean_cols], 
                           use = "pairwise.complete.obs")

print(round(co_implementation_matrix, 2))

In [ ]:
head(df_demeaned)

## Visualization

In [ ]:
# 1. Melt the matrix into a long format for ggplot
melted_matrix <- melt(co_implementation_matrix)

# 2. Plot the heatmap
ggplot(data = melted_matrix, aes(x = Var1, y = Var2, fill = value)) +
  geom_tile(color = "white") + # Adds thin borders between tiles
  scale_fill_gradient2(low = "blue", high = "red", mid = "white", 
                       midpoint = 0, limit = c(-1, 1), space = "Lab", 
                       name="Within-Repo\nCorrelation") +
  geom_text(aes(label = round(value, 2)), color = "black", size = 3) + # Overlays numbers
  theme_minimal() + 
  theme(axis.text.x = element_text(angle = 45, vjust = 1, hjust = 1), # Rotates labels
        axis.title.x = element_blank(),
        axis.title.y = element_blank(),
        panel.grid.major = element_blank()) +
  coord_fixed() # Keeps the tiles perfectly square